# Tutorial: MiniAutoGen e a Nova Arquitetura de Drivers

## Público
- Time de engenharia que vai implementar, manter ou integrar o MiniAutoGen.

## Pré-requisitos
- Python assíncrono (`async` / `await`).
- Binário `gemini` instalado e autenticado localmente.
- Dependências do projeto disponíveis no ambiente do notebook.

## Objetivos
- Entender a arquitetura de **backend drivers** (`AgentAPIDriver`, `BackendResolver`).
- Executar o runtime oficial com gateway local embutido (sem servidor externo).
- Ver contratos, stores, driver e runner funcionando juntos.
- Aprender o padrão de consumo de eventos via `async for event in driver.send_turn(...)`.

## Roteiro

1. Setup do notebook
2. Pré-checagem do Gemini CLI
3. Contratos centrais (core + backend models)
4. Stores mínimos
5. Driver real via Gemini CLI Gateway (`AgentAPIDriver`)
6. BackendResolver + factory pattern
7. Runtime oficial com `PipelineRunner`
8. Exemplo integrado de ponta a ponta
9. Checklist de caminho novo
10. Exercício

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import httpx


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    return start

repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

repo_root

## 1. Pré-checagem do motor LLM

Antes de qualquer exemplo, o notebook valida se o `gemini` está instalado e autenticado. Isso evita notebooks “verdes” com provider fake.


In [ ]:
def assert_gemini_cli_ready() -> dict[str, object]:
    binary = subprocess.run(
        ['bash', '-lc', 'command -v gemini'],
        capture_output=True,
        text=True,
        check=False,
    )
    gemini_path = binary.stdout.strip()
    if not gemini_path:
        raise RuntimeError('Binary `gemini` não encontrado no PATH.')

    last_error = None
    for attempt in range(1, 4):
        try:
            smoke = subprocess.run(
                ['gemini', '-m', 'gemini-2.5-pro', '--output-format', 'json'],
                input='Respond with exactly: NOTEBOOK-GEMINI-OK',
                capture_output=True,
                text=True,
                check=False,
                timeout=180,
            )
            if smoke.returncode != 0:
                last_error = f'Gemini CLI falhou no smoke test: {smoke.stderr}'
                continue
            if 'NOTEBOOK-GEMINI-OK' not in smoke.stdout:
                last_error = (
                    'Smoke test do Gemini CLI não retornou o texto esperado. '
                    f'stdout={smoke.stdout!r}'
                )
                continue
            return {
                'gemini_path': gemini_path,
                'smoke_test': 'NOTEBOOK-GEMINI-OK',
                'attempts_used': attempt,
                'stderr_excerpt': smoke.stderr.strip()[:200],
            }
        except subprocess.TimeoutExpired as exc:
            last_error = f'Timeout no smoke test do Gemini CLI após {exc.timeout} segundos.'

    raise RuntimeError(last_error or 'Falha desconhecida no smoke test do Gemini CLI.')

assert_gemini_cli_ready()


## 2. Contratos centrais

A arquitetura é baseada em dois níveis de contratos:

### Core contracts (estáveis)
- `Message`, `ExecutionEvent`, `RunContext` — modelos do runtime.
- `ExecutionPolicy` — configuração de timeout e comportamento.

### Backend models (nova camada de drivers)
- `AgentEvent` — evento canônico emitido por qualquer backend.
- `BackendCapabilities` — declara o que um backend suporta (streaming, cancel, tools...).
- `StartSessionRequest` / `StartSessionResponse` — protocolo de sessão.
- `SendTurnRequest` — envio de um turno (mensagens) ao backend.
- `BackendConfig` / `DriverType` — configuração declarativa de backends.

In [ ]:
from miniautogen.core.contracts import ExecutionEvent, Message, RunContext
from miniautogen.policies import ExecutionPolicy

# --- Core contracts ---
message = Message(sender_id='agent_1', content='hello from driver architecture')
event = ExecutionEvent(type='run_started', correlation_id='corr-001', payload={'stage': 'tutorial'})
context = RunContext(
    run_id='tutorial-run-001',
    started_at=datetime.now(timezone.utc),
    correlation_id='corr-tutorial-001',
    input_payload={'prompt': 'Explique a arquitetura em uma frase.'},
)

# --- Backend models ---
from miniautogen.backends.models import (
    AgentEvent,
    BackendCapabilities,
    SendTurnRequest,
    StartSessionRequest,
    StartSessionResponse,
)
from miniautogen.backends.config import BackendConfig, DriverType

caps = BackendCapabilities(sessions=False, streaming=False, cancel=False)
session_req = StartSessionRequest(backend_id='gemini', system_prompt='Você é um assistente útil.')
turn_req = SendTurnRequest(
    session_id='sess_example',
    messages=[{'role': 'user', 'content': 'Olá!'}],
)
agent_event = AgentEvent(type='message_completed', session_id='sess_example', turn_id='turn_001', payload={'text': 'Oi!'})

{
    'core': {
        'message': message.model_dump(),
        'event': event.model_dump(),
        'context': context.model_dump(),
        'execution_policy': ExecutionPolicy(timeout_seconds=30.0),
    },
    'backend': {
        'capabilities': caps.model_dump(),
        'capabilities_supported': caps.supported(),
        'session_request': session_req.model_dump(),
        'turn_request': turn_req.model_dump(),
        'agent_event': agent_event.model_dump(),
    },
}

## 3. Stores mínimos

Os stores continuam separados por responsabilidade. Neste tutorial, vamos exercitar mensagens, runs e checkpoints antes de conectar o driver real.

In [ ]:
from miniautogen.stores import InMemoryCheckpointStore, InMemoryMessageStore, InMemoryRunStore

message_store = InMemoryMessageStore()
run_store = InMemoryRunStore()
checkpoint_store = InMemoryCheckpointStore()

await message_store.add_message(Message(sender_id='user', content='ping'))
await run_store.save_run('run-1', {'status': 'started'})
await checkpoint_store.save_checkpoint('run-1', {'step': 'bootstrap'})

{
    'messages': [m.model_dump() for m in await message_store.get_messages()],
    'run': await run_store.get_run('run-1'),
    'checkpoint': await checkpoint_store.get_checkpoint('run-1'),
}


## 4. Driver real via Gemini CLI Gateway (`AgentAPIDriver`)

O notebook não fala com `gemini` diretamente. Ele usa o `AgentAPIDriver`, que fala com o gateway ASGI local via HTTP embutido (sem servidor externo).

### Arquitetura em camadas
```
Notebook → AgentAPIDriver → AgentAPIClient → httpx.ASGITransport → gemini_cli_gateway → gemini CLI
```

O `AgentAPIClient` cuida de retry, timeout e health check. O `AgentAPIDriver` implementa o contrato `AgentDriver` e converte respostas HTTP em `AgentEvent`s canônicos.

In [ ]:
from gemini_cli_gateway.app import app as gateway_app

from miniautogen.backends.agentapi import AgentAPIClient, AgentAPIDriver
from miniautogen.backends.models import SendTurnRequest, StartSessionRequest

# 1. Criar client HTTP apontando para o gateway ASGI embutido
client = AgentAPIClient(
    base_url='http://gemini-gateway',
    transport=httpx.ASGITransport(app=gateway_app),
    health_endpoint=None,       # gateway embutido não precisa de health check
    max_retry_attempts=3,
    timeout_seconds=90.0,
)

# 2. Criar driver que implementa o contrato AgentDriver
driver = AgentAPIDriver(client=client, model='gemini-2.5-pro')

# 3. Iniciar sessão
session = await driver.start_session(StartSessionRequest(backend_id='gemini'))

{
    'driver_class': driver.__class__.__name__,
    'session_id': session.session_id,
    'capabilities': session.capabilities.model_dump(),
}

In [ ]:
# Helper para extrair texto simples do stream de eventos
async def ask_model(prompt: str, system_prompt: str | None = None) -> str:
    """Envia um turno ao driver e retorna o texto da resposta."""
    messages = []
    if system_prompt:
        messages.append({'role': 'system', 'content': system_prompt})
    messages.append({'role': 'user', 'content': prompt})

    async for event in driver.send_turn(
        SendTurnRequest(session_id=session.session_id, messages=messages)
    ):
        if event.type == 'message_completed':
            return event.payload.get('text', '')
    return ''


# Testar o driver com o Gemini CLI real
driver_response = await ask_model('Respond with exactly: TUTORIAL-DRIVER-OK')
driver_response

## 5. BackendResolver + factory pattern

Para cenários com múltiplos backends (Gemini, OpenAI, Ollama...), o `BackendResolver` resolve um `backend_id` em um driver configurado. A factory `agentapi_factory` cria o driver a partir de um `BackendConfig` declarativo.

Este é o padrão recomendado para aplicações de produção — o driver é criado e cacheado automaticamente.

In [ ]:
from miniautogen.backends import BackendResolver, agentapi_factory
from miniautogen.backends.config import BackendConfig, DriverType

# 1. Criar resolver e registrar a factory do AgentAPI
resolver = BackendResolver()
resolver.register_factory(DriverType.AGENT_API, agentapi_factory)

# 2. Adicionar configuração declarativa do backend Gemini
resolver.add_backend(BackendConfig(
    backend_id='gemini-via-resolver',
    driver=DriverType.AGENT_API,
    endpoint='http://gemini-gateway',
    timeout_seconds=90.0,
    metadata={
        'model': 'gemini-2.5-pro',
        'health_endpoint': None,    # gateway embutido
        'max_retry_attempts': 3,
    },
))

# 3. Resolver o driver — criado e cacheado automaticamente
resolved_driver = resolver.get_driver('gemini-via-resolver')

{
    'backends_configurados': resolver.list_backends(),
    'driver_class': resolved_driver.__class__.__name__,
    'config': resolver.get_config('gemini-via-resolver').model_dump(),
}

## 6. Runtime oficial com `PipelineRunner`

O `PipelineRunner` continua sendo o caminho oficial para executar pipelines. A diferença é que agora a pipeline usa o `AgentAPIDriver` internamente, consumindo eventos via `async for`.

In [ ]:
from miniautogen.core.events import InMemoryEventSink
from miniautogen.core.runtime import PipelineRunner
from miniautogen.stores import InMemoryCheckpointStore, InMemoryRunStore


class DriverBackedPipeline:
    """Pipeline que usa o AgentAPIDriver para gerar respostas."""

    def __init__(self, driver, session_id: str):
        self.driver = driver
        self.session_id = session_id

    async def run(self, state):
        prompt = state['prompt']
        messages = [{'role': 'user', 'content': prompt}]
        reply = ''
        async for event in self.driver.send_turn(
            SendTurnRequest(session_id=self.session_id, messages=messages)
        ):
            if event.type == 'message_completed':
                reply = event.payload.get('text', '')
        return {'reply': reply, 'prompt': prompt}


sink = InMemoryEventSink()
runner = PipelineRunner(
    event_sink=sink,
    run_store=InMemoryRunStore(),
    checkpoint_store=InMemoryCheckpointStore(),
    execution_policy=ExecutionPolicy(timeout_seconds=90.0),
)

runtime_result = await runner.run_pipeline(
    DriverBackedPipeline(driver, session.session_id),
    {'prompt': 'Responda exatamente: TUTORIAL-RUNNER-OK'},
)

{
    'result': runtime_result,
    'run_id': runner.last_run_id,
    'events': [event.type for event in sink.events],
}

## 7. Exemplo integrado de ponta a ponta

Aqui juntamos driver real, runtime, stores e mensagens. O objetivo é mostrar o fluxo novo inteiro: sessão, turno, eventos, persistência.

In [ ]:
integrated_message_store = InMemoryMessageStore()
integrated_run_store = InMemoryRunStore()
integrated_checkpoint_store = InMemoryCheckpointStore()


class FullPipeline:
    """Pipeline completa: driver + message store + eventos."""

    def __init__(self, driver, session_id: str, message_store):
        self.driver = driver
        self.session_id = session_id
        self.message_store = message_store

    async def run(self, state):
        prompt = state['prompt']
        await self.message_store.add_message(Message(sender_id='user', content=prompt))

        messages = [{'role': 'user', 'content': prompt}]
        reply = ''
        all_events = []

        async for event in self.driver.send_turn(
            SendTurnRequest(session_id=self.session_id, messages=messages)
        ):
            all_events.append(event)
            if event.type == 'message_completed':
                reply = event.payload.get('text', '')

        await self.message_store.add_message(Message(sender_id='assistant', content=reply))
        return {'reply': reply, 'agent_events': [e.type for e in all_events]}


full_runner = PipelineRunner(
    run_store=integrated_run_store,
    checkpoint_store=integrated_checkpoint_store,
    execution_policy=ExecutionPolicy(timeout_seconds=90.0),
)

integrated_result = await full_runner.run_pipeline(
    FullPipeline(driver, session.session_id, integrated_message_store),
    {'prompt': 'Responda exatamente: TUTORIAL-END-TO-END-OK'},
)

{
    'result': integrated_result,
    'messages': [m.model_dump() for m in await integrated_message_store.get_messages()],
    'run': await integrated_run_store.get_run(full_runner.last_run_id),
    'checkpoint': await integrated_checkpoint_store.get_checkpoint(full_runner.last_run_id),
}

## 8. Caminho novo vs legado

Use esta regra simples ao evoluir o projeto:

| Camada | Caminho novo | Legado (compat) |
|--------|-------------|-----------------|
| **LLM** | `backends.agentapi.AgentAPIDriver` | `adapters.llm.OpenAICompatibleProvider` |
| **Config** | `backends.config.BackendConfig` + `BackendResolver` | Construção manual |
| **Eventos** | `AgentEvent` via `async for event in driver.send_turn(...)` | String retornada por `provider.generate_response()` |
| **Runtime** | `core.runtime.PipelineRunner` | Igual (estável) |
| **Stores** | `stores.*` | Igual (estável) |

### Resumo da migração
1. Trocar `OpenAICompatibleProvider` por `AgentAPIClient` + `AgentAPIDriver`
2. Usar `StartSessionRequest` para iniciar sessão
3. Consumir eventos com `async for event in driver.send_turn(...)`
4. Para multi-backend: usar `BackendResolver` + `BackendConfig` + `agentapi_factory`

## Exercício

Implemente uma pipeline que:
1. Receba um prompt no state
2. Persista a entrada no message store
3. Use o `AgentAPIDriver` via `send_turn()` para obter a resposta
4. Colete todos os `AgentEvent`s emitidos durante o turno
5. Devolva a resposta junto com `summary_length` e a lista de tipos de evento

In [ ]:
class ExercisePipeline:
    def __init__(self, driver, session_id: str, message_store):
        self.driver = driver
        self.session_id = session_id
        self.message_store = message_store

    async def run(self, state):
        prompt = state['prompt']
        await self.message_store.add_message(Message(sender_id='user', content=prompt))

        messages = [{'role': 'user', 'content': prompt}]
        reply = ''
        event_types = []

        async for event in self.driver.send_turn(
            SendTurnRequest(session_id=self.session_id, messages=messages)
        ):
            event_types.append(event.type)
            if event.type == 'message_completed':
                reply = event.payload.get('text', '')

        await self.message_store.add_message(Message(sender_id='assistant', content=reply))
        return {'reply': reply, 'summary_length': len(reply), 'event_types': event_types}


# Exemplo de uso:
# exercise_runner = PipelineRunner(
#     run_store=InMemoryRunStore(),
#     checkpoint_store=InMemoryCheckpointStore(),
#     execution_policy=ExecutionPolicy(timeout_seconds=90.0),
# )
# await exercise_runner.run_pipeline(
#     ExercisePipeline(driver, session.session_id, InMemoryMessageStore()),
#     {'prompt': 'Resuma a arquitetura em 5 palavras.'},
# )

In [ ]:
await client.close()
'driver client closed'